In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
import os
import subprocess
import sys
from unittest.mock import patch

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
conf_path = os.path.abspath("log4j2.xml")

def _silent_popen(*args, **kwargs):
    return subprocess.Popen(*args, **kwargs, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("MySparkProject")
    .config("spark.ui.showConsoleProgress", "false")
    .config(
        "spark.driver.extraJavaOptions",
        f"-Dlog4j.configurationFile={conf_path} -Divy.message.logger.level=0",
    )
)

with patch("pyspark.java_gateway.Popen", side_effect=_silent_popen):
    spark = builder.getOrCreate()

#spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version} is ready.")
print(f"Java version: {spark.sparkContext._jvm.java.lang.System.getProperty('java.version')}")
print(f"Python: {sys.version.split(' ')[0]}")
print(f"Hadoop: {spark.sparkContext._gateway.jvm.org.apache.hadoop.util.VersionInfo.getVersion()}")

Spark 4.1.1 is ready.
Java version: 21.0.9
Python: 3.10.12
Hadoop: 3.4.2


In [2]:
book = spark.read.text("../data/gutenberg_books/1342-0.txt")

In [3]:
book

value
The Project Guten...
""
This eBook is for...
almost no restric...
re-use it under t...
with this eBook o...
""
""
Title: Pride and ...
""


In [4]:
book.printSchema()

root
 |-- value: string (nullable = true)



In [5]:
print(book.dtypes)

[('value', 'string')]


In [6]:
book.show()

+--------------------+
|               value|
+--------------------+
|The Project Guten...|
|                    |
|This eBook is for...|
|almost no restric...|
|re-use it under t...|
|with this eBook o...|
|                    |
|                    |
|Title: Pride and ...|
|                    |
| Author: Jane Austen|
|                    |
|Posting Date: Aug...|
|Release Date: Jun...|
|Last Updated: Mar...|
|                    |
|   Language: English|
|                    |
|Character set enc...|
|                    |
+--------------------+
only showing top 20 rows


In [7]:
book.show(10, truncate=50)

+--------------------------------------------------+
|                                             value|
+--------------------------------------------------+
|The Project Gutenberg EBook of Pride and Prejud...|
|                                                  |
|This eBook is for the use of anyone anywhere at...|
|almost no restrictions whatsoever.  You may cop...|
|re-use it under the terms of the Project Gutenb...|
|    with this eBook or online at www.gutenberg.org|
|                                                  |
|                                                  |
|                        Title: Pride and Prejudice|
|                                                  |
+--------------------------------------------------+
only showing top 10 rows


In [8]:
lines = book.select(F.split("value", " ").alias("line"))

In [9]:
lines.show(5)

+--------------------+
|                line|
+--------------------+
|[The, Project, Gu...|
|                  []|
|[This, eBook, is,...|
|[almost, no, rest...|
|[re-use, it, unde...|
+--------------------+
only showing top 5 rows


In [10]:
lines.printSchema()

root
 |-- line: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [11]:
words = lines.select(F.explode("line").alias("word"))

In [12]:
words.show(15)

+----------+
|      word|
+----------+
|       The|
|   Project|
| Gutenberg|
|     EBook|
|        of|
|     Pride|
|       and|
|Prejudice,|
|        by|
|      Jane|
|    Austen|
|          |
|      This|
|     eBook|
|        is|
+----------+
only showing top 15 rows


In [13]:
words_lower = words.select(F.lower("word").alias("word_lower"))

In [14]:
words_lower.show(10)

+----------+
|word_lower|
+----------+
|       the|
|   project|
| gutenberg|
|     ebook|
|        of|
|     pride|
|       and|
|prejudice,|
|        by|
|      jane|
+----------+
only showing top 10 rows


In [15]:
words_clean = words_lower.select(F.regexp_extract("word_lower", "[a-z]+", 0).alias("word"))

In [16]:
words_clean.show(20)

+---------+
|     word|
+---------+
|      the|
|  project|
|gutenberg|
|    ebook|
|       of|
|    pride|
|      and|
|prejudice|
|       by|
|     jane|
|   austen|
|         |
|     this|
|    ebook|
|       is|
|      for|
|      the|
|      use|
|       of|
|   anyone|
+---------+
only showing top 20 rows


In [17]:
words_notnull = words_clean.filter(F.col("word") != "")

In [18]:
words_notnull.show(20)

+---------+
|     word|
+---------+
|      the|
|  project|
|gutenberg|
|    ebook|
|       of|
|    pride|
|      and|
|prejudice|
|       by|
|     jane|
|   austen|
|     this|
|    ebook|
|       is|
|      for|
|      the|
|      use|
|       of|
|   anyone|
| anywhere|
+---------+
only showing top 20 rows


In [19]:
results = words_notnull.groupBy("word").count()

In [22]:
results.show(10)

+---------+-----+
|     word|count|
+---------+-----+
|   online|    4|
|     some|  209|
|    still|   72|
|      few|   72|
|     hope|  122|
|    those|   60|
| cautious|    4|
|imitation|    1|
|      art|    3|
|  solaced|    1|
+---------+-----+
only showing top 10 rows


In [23]:
results.orderBy("count", ascending=False).show(10)

+----+-----+
|word|count|
+----+-----+
| the| 4496|
|  to| 4235|
|  of| 3719|
| and| 3602|
| her| 2223|
|   i| 2052|
|   a| 1997|
|  in| 1920|
| was| 1844|
| she| 1703|
+----+-----+
only showing top 10 rows


In [24]:
%sql

UsageError: Line magic function `%sql` not found.
